# TFLite Export — ASL Sign Language Model
Standalone notebook. **No need to re-run training notebooks.**  
Loads best model (`EfficientNetB0 fine-tuned`) directly from Google Drive and exports TFLite variants for deployment on Raspberry Pi.

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Imports & Dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

# Download test split from Kaggle (for INT8 calibration)
if not os.path.exists("sign_mnist_test.csv") and not os.path.exists("sign_mnist_test/sign_mnist_test.csv"):
    os.environ['KAGGLE_USERNAME'] = "abdullahashiry"
    os.environ['KAGGLE_KEY']      = "KGAT_331632d901a6cb7a05431b55135bd8c2"
    !pip install -q kaggle
    !kaggle datasets download -d datamunge/sign-language-mnist --unzip

test_path = ("sign_mnist_test/sign_mnist_test.csv"
             if os.path.exists("sign_mnist_test/sign_mnist_test.csv")
             else "sign_mnist_test.csv")

test_df = pd.read_csv(test_path)
x_test  = test_df.drop("label", axis=1).values.reshape(-1, 28, 28, 1).astype("float32") / 255.0
print(f"Test set loaded: {x_test.shape}")

## 2. Load Trained Model

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/CV552_SignLanguage/models/efficientnetb0_finetuned.keras"
model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

## 3. Convert to TFLite (Float32)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("asl_model.tflite", "wb") as f:
    f.write(tflite_model)
print("Saved: asl_model.tflite")

## 4. Convert to TFLite (INT8 Quantized — smaller + faster on Pi)

In [ ]:
def representative_dataset():
    for i in range(min(200, len(x_test))):
        yield [x_test[i:i+1]]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type  = tf.float32
converter_int8.inference_output_type = tf.float32

tflite_model_int8 = converter_int8.convert()

with open("asl_model_int8.tflite", "wb") as f:
    f.write(tflite_model_int8)
print("Saved: asl_model_int8.tflite")

## 5. Sanity Check — Float32 Model

In [ ]:
interpreter = tf.lite.Interpreter(model_path="asl_model.tflite")
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input shape  expected:", input_details[0]['shape'])   # [1, 28, 28, 1]
print("Output shape expected:", output_details[0]['shape'])  # [1, 25]

dummy = np.zeros((1, 28, 28, 1), dtype=np.float32)
interpreter.set_tensor(input_details[0]['index'], dummy)
interpreter.invoke()
out = interpreter.get_tensor(output_details[0]['index'])
print("Output (dummy):", out)

## 6. Model Size Comparison

In [ ]:
size_keras = os.path.getsize(MODEL_PATH) / 1e6
size_fp32  = os.path.getsize("asl_model.tflite") / 1e6
size_int8  = os.path.getsize("asl_model_int8.tflite") / 1e6

print(f"Model Size Comparison:")
print(f"  Original .keras : {size_keras:.2f} MB")
print(f"  TFLite FP32     : {size_fp32:.2f} MB")
print(f"  TFLite INT8     : {size_int8:.2f} MB")

## 7. Copy Models to Drive (optional)

In [ ]:
import shutil

DRIVE_OUT = "/content/drive/MyDrive/CV552_SignLanguage/tflite"
os.makedirs(DRIVE_OUT, exist_ok=True)

shutil.copy("asl_model.tflite",      os.path.join(DRIVE_OUT, "asl_model.tflite"))
shutil.copy("asl_model_int8.tflite", os.path.join(DRIVE_OUT, "asl_model_int8.tflite"))
print(f"Copied both .tflite files to {DRIVE_OUT}")